In [81]:
import sys
import os
sys.path.append(os.path.abspath('../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.repeated_cv import *
from functions.new_variables import *
from scipy.stats import skew, shapiro

In [82]:
dataset = pd.read_excel('../../../BreastCareCond.xlsx')
data_code = pd.read_excel('../../../Codificação variaveis.xlsx')

In [83]:
#Defining the proposed has good treatment variables to use (the empty ones were removed)
relevant_variables = ['PERSON_ID', 'EBC', 'Final pathology']
numerical_variables = ['Age', 'Ki67 positive cells (%) at CB']
categorical_variables = ['N (Regional nodes affected)', 'Number identifying the lesion in the patient', 
                         'Side location of the lesion', 'Classification with respect to other lesions', 
                         'Source of referral', 'Type of invasion at CB', 'Isotype at CB', 
                         'Oestrogen receptor status at CB', 'Progesterone receptor status at CB', 
                         'Disease extent', 'Grade at CB', 'Her2 overexpression (with immunohystochemistry) at CB']

In [84]:
#correcting th error in the column name
dataset.rename(columns={'Istotype at CB': 'Isotype at CB'}, inplace=True)

In [85]:
dataset = filter_EBC(dataset)
dataset.reset_index(drop=True, inplace=True)

In [86]:
dataset = dataset[dataset['Final recommendations to patient'] == 4]
dataset = dataset[dataset['Final pathology'].isin([1.0, 4.0])]
dataset.reset_index(drop=True, inplace=True)

In [87]:
#since the clinical staging uses the nodes
node = ['N (Regional nodes affected)']
fill_with_mode(dataset, node)
cT(dataset)
clinical_staging (dataset)

# pT was already loaded
# e vou cagar na variavel do cT fica só o outro T by imaging or clinical examination
new_variables = ['Cancer stage']
categorical_variables = categorical_variables + new_variables

In [88]:
change_to_numeric(dataset, numerical_variables)
change_to_categorical(dataset, categorical_variables)

Done
Done


In [89]:
columns_to_map = {
    'N (Regional nodes affected)': {0: 0, 1: 1, 2: 2_3, 3: 2_3},
    'Side location of the lesion': {'L': 0, 'R': 1},
    'Source of referral': {1: 0, 2: 1},
    'Type of invasion at CB': {1: 0, 2: 1},
    'Oestrogen receptor status at CB': {1: 0, 2: 1},
    'Progesterone receptor status at CB': {1: 0, 2: 1},
    'Number identifying the lesion in the patient': {1: 1, 2: 0, 3: 0, 4: 0},
    'Isotype at CB': {1: 1, 2: 2, 3: 4, 4: 4},
    'Disease extent': {0: 0, 1: 1, 2: 1},
    #mapping the target
    'Final pathology': {1.0: 1, 4.0: 0}
}

for column, mapping in columns_to_map.items():
    dataset[column] = dataset[column].map(mapping)

In [90]:
# Replace missing values in the specified columns
mode_variables = ['Side location of the lesion', 'Classification with respect to other lesions', 'Type of invasion at CB', 'Disease extent', 
                'Oestrogen receptor status at CB', 'Cancer stage', 'Isotype at CB', 'Progesterone receptor status at CB', 
                'Grade at CB', 'Her2 overexpression (with immunohystochemistry) at CB']
fill_with_mode(dataset, mode_variables)

'Done'

In [91]:
ki67 = ['Ki67 positive cells (%) at CB']
median_imputer(dataset, ki67)

'Done'

In [92]:
one_hot_variables = ['N (Regional nodes affected)', 'Classification with respect to other lesions', 'Isotype at CB', 'Cancer stage', 'Grade at CB', 'Her2 overexpression (with immunohystochemistry) at CB']
ohe_dataset, new_column_names = one_hot_encoding(dataset, one_hot_variables, drop_first=None)

In [93]:
categorical_variables = [col for col in categorical_variables if col not in one_hot_variables]
categorical_variables.extend(new_column_names.tolist())

selected_variables = relevant_variables + numerical_variables + categorical_variables

In [94]:
treatment_variables = ['Radiotherapy (RT) performed', 'Conventional RT fraction', 'Total administered dose', 'Boost dose administered', 
                       'Radiotherapy on chest wall', 'Radiotherapy on supraclavicular area', 'Radiotherapy on internal mammary chain',
                         'State (if hormone therapy prescribed)', 'Treatment in association with chemotherapy', 
                        'Chemotherapy performed', 'Adjuvant chemotherapy', 'Adjuvant therapy with biological drugs', 'Endocrine therapy performed']

dataset = ohe_dataset

In [95]:
numerical_data, categorical_data = type_separation(data_code, treatment_variables)
len(numerical_data), len(categorical_data)

(3, 10)

In [96]:
replace_missing_values(dataset, numerical_data, 0)

Done


In [97]:
dataset.drop(index=0, inplace=True)
dataset.reset_index(inplace=True, drop=True)
replace_missing_values(dataset, categorical_data, 0)

Done


In [98]:
variable_two_to_zero = ['Adjuvant chemotherapy', 'Chemotherapy performed', 'Endocrine therapy performed']
variable_two_to_one = ['Adjuvant therapy with biological drugs']

two_to_zero(dataset, variable_two_to_zero)
two_to_one(dataset, variable_two_to_one)

'Done'

In [99]:
to_remove = ['PERSON_ID', 'EBC', 'Final pathology']
important_variables = ['PERSON_ID', 'Final pathology']
diagnosis_variables = [col for col in selected_variables if col not in to_remove]
selected_columns = important_variables + treatment_variables + diagnosis_variables

In [100]:
survival_dataset = dataset[selected_columns]

In [101]:
# Check if there are any NaN values in the dataset
contains_nan = survival_dataset.isnull().any().any()
print("Does survival_dataset contain NaN values?", contains_nan)

# Optionally, to see which columns contain NaN values
nan_columns = survival_dataset.columns[survival_dataset.isnull().any()].tolist()
print("Columns with NaN values:", nan_columns)

Does survival_dataset contain NaN values? False
Columns with NaN values: []


In [102]:
survival_dataset.to_csv("data.csv")

In [103]:
with open("diagnosis.txt", "w") as file:
    for item in diagnosis_variables:
        file.write(str(item) + "\n")
with open("treatment.txt", "w") as file:
    for item in treatment_variables:
        file.write(str(item) + "\n")

In [106]:
survival_dataset['Final pathology'].value_counts()

Final pathology
0    931
1    356
Name: count, dtype: int64